**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Capstone: Build a Full System

Not new theory — proof that the curriculum *composes*. One pipeline, end to end: IQ capture → detection (matched filter + CFAR) → tracking (Kalman) → classification (CNN) → results database. Every stage is a workshop you've taken; here they hold hands. Runs on synthetic IQ so it executes anywhere; swap in an [RTL-SDR capture](../Intro_SDR/Software_Defined_Radio.ipynb) and nothing else changes.

## 1. Pre-requisites

The whole curriculum, honestly — minimally: [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb), [Kalman](../Intro_Time_Series/Intro_AdFilt_KF.ipynb), [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb), [Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import sqlite3
import matplotlib.pyplot as plt
from scipy import signal as sig
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *The Scenario & the Signal Generator* (~30 min)
**Goal:** an emitter moves through a noisy band, transmitting bursts; we build the world to be monitored.
**Feeds into:** Session 2 (detection).

---

## 2. The World

💡 **Intuition.** System design starts with the truth you'll grade yourself against. Our world: an emitter drifts across the band (a slowly-moving carrier frequency), transmitting short bursts of one of three modulation types, buried in noise. The pipeline must find the bursts, track the drift, and identify the modulation — and because we built the world, every stage gets an oracle.

In [ ]:
# emitter truth: carrier random-walks; burst type fixed per emitter

# YOUR CODE HERE


**What just happened.** `burst_starts` planted 55 bursts across the 4-second window (one every `BURST+GAP` = 0.07 s, minus edge margins), and the printed carrier range (13.5–35.1 kHz) is the realized excursion of a random walk that started at 20 kHz — Gaussian steps of $\sigma=30$ Hz accumulate over 400,000 samples, so the carrier is free to drift several kHz in either direction over the full recording. This random-walk carrier is deliberately the hard part of the scenario: every downstream stage has to cope with a target that doesn't sit still in frequency, which is exactly why Session 3 needs a Kalman filter rather than a fixed frequency estimate.

---
### 🕐 Session 2 of 4 — *Detection: Energy → CFAR* (~35 min)
**Goal:** find the bursts in time-frequency; CFAR keeps false alarms honest.
**Builds on:** [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S4; [Radar](../Intro_DSP/Radar_Signal_Processing.ipynb) S3. &nbsp; **Feeds into:** Session 3 (tracking).

---

In [ ]:
# collapse to a detection statistic per frame: max power over frequency
# CA-CFAR along time (from the Radar workshop, verbatim idea)
# group consecutive detections into events; record their peak frequency

# YOUR CODE HERE


**What just happened.** CFAR found all 55 planted bursts with zero missed detections and zero spurious events (55 detected events, 100% hit rate) — a clean result that reflects a genuinely easy detection problem: each burst is injected at amplitude 4.0 against a unit-variance noise floor, a very high SNR by design, so this scenario is testing the *pipeline plumbing*, not detector sensitivity at the edge of noise. The median-based CA-CFAR threshold (`scale*median(ref)` rather than the more common mean-based estimator) is the deliberate choice here: a mean-based reference window can itself get pulled upward by a burst sitting inside the reference cells, inflating the threshold and causing a miss right next to a strong signal — the median is far more robust to exactly that kind of self-pollution, which matters once bursts start arriving close together in time.

---
### 🕐 Session 3 of 4 — *Tracking the Drift: Kalman* (~35 min)
**Goal:** the detections are noisy frequency snapshots; a Kalman filter strings them into a track.
**Builds on:** [Kalman workshop](../Intro_Time_Series/Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 4 (classification & the database).

---

In [ ]:
# constant-velocity Kalman on (frequency, drift-rate), fed by the detected events

# YOUR CODE HERE


**What just happened.** The plot title reports the raw per-detection frequency error against the Kalman-smoothed track's error — the Kalman filter should show a clear RMSE reduction, and here's why: each individual CFAR detection's peak-frequency estimate is quantized by the STFT's frequency-bin resolution (`fs/nperseg` ≈ 98 Hz per bin at `nperseg=1024`) and corrupted by that frame's noise realization, while the Kalman filter combines the constant-velocity motion model (`F`) with every measurement seen so far, averaging down noise that's independent frame-to-frame while still tracking genuine drift via the velocity state. The process noise `Q` and measurement noise `R` set how much the filter trusts the model vs. the data — `Q`'s large drift-rate entry (3000) tells the filter the carrier's velocity can itself change quickly (matching the random-walk carrier), while `R` (80,000 Hz²) encodes how noisy a single CFAR frequency estimate is expected to be.

---
### 🕐 Session 4 of 4 — *Classification & the Ledger* (~40 min)
**Goal:** a CNN identifies each burst's modulation; every verdict lands in a queryable database.
**Builds on:** [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb); [Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb).

---

In [ ]:
# train the classifier on synthetic bursts of the 3 candidate types (baseband spectrograms)
# run it on the DETECTED bursts (mix each event down to baseband using the KALMAN track)

# YOUR CODE HERE


**What just happened — and it's worth reading past the headline number.** The CNN hit 100% holdout accuracy on synthetic training-distribution bursts, but every one of the 55 real bursts in this scenario is planted as `chirp` — and the pipeline's own votes split 32 chirp / 19 bpsk / 4 fm, only ~58% "chirp." That gap between 100% holdout accuracy and ~58% correct-on-deployment is not a contradiction, it's a real and common failure mode: the *training* distribution (`burst_example`) adds a random residual frequency offset (`df`, $\sigma=150$ Hz) to simulate tracking error, but the *actual* bursts fed to the classifier are downmixed using the Kalman-tracked frequency, whose true residual error depends on the track quality computed two cells up — if that residual error, timing alignment, or noise realization differs from what `burst_example` simulated, the classifier is evaluating on a distribution it wasn't quite trained for. The majority vote still recovers the correct system-level verdict ('chirp', matching planted truth) because 32 is a plurality, but the *confidence* column tells the same story: chirp's mean confidence (0.69) is actually *lower* than the wrong-every-time bpsk verdict's (0.78) — a reminder that a per-detection classifier being right on average doesn't mean any single verdict, or its confidence score, should be trusted at face value. That's exactly why the ledger stores every vote instead of collapsing straight to one number.

## 3. Conclusion

Detection found the bursts (hit rate printed), the Kalman track cut the frequency error and *steered the downmixer*, the CNN identified the modulation, and SQL holds the evidence. No stage is new; the composition is the achievement — and every interface between stages is a place your own projects can swap in smarter pieces.

**The real-hardware version:** replace Session 1 with an [RTL-SDR capture](../Intro_SDR/Software_Defined_Radio.ipynb); budget the pipeline with [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb); containerize it with [Containers](../Intro_Host_Prog/Intro_Containers/Intro_Containers.ipynb). That's a senior-project-grade system, from parts you already own.

---
**This is the final playlist.** Where next is yours.